# Churn Data Cleaning for Automated Retraining

This notebook cleans `data/raw/Telco_customer_churn.xlsx` into a model-ready feature matrix `X` and target vector `y` suitable for an automated churn-retraining pipeline.

**Convention used throughout:** every cleaning decision is documented in a comment block that states:
1. **Decision** — what was done.
2. **Alternatives considered** — other plausible options.
3. **Why not picked** — rationale for rejecting each alternative.

Only decisions that affect the retraining inputs are commented in depth; exploratory prints are kept minimal.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display full width for inspection output
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

DATA_PATH = Path("data/raw/Telco_customer_churn.xlsx")
CLEAN_PATH = Path("data/processed/Telco_customer_churn_cleaned.csv")
FEATURES_PATH = Path("data/processed/X_features.csv")
TARGET_PATH = Path("data/processed/y_target.csv")

print("Imports ready.")

In [ ]:
df = pd.read_excel(DATA_PATH)
print("Loaded shape:", df.shape)
print("Columns:", list(df.columns))


In [ ]:
print("=== info ===")
print(df.info())
print()
print("=== missing values (only columns with nulls) ===")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print("=== duplicates ===")
print("duplicate CustomerID:", df["CustomerID"].duplicated().sum())
print("duplicate full rows:", df.duplicated().sum())


In [ ]:
# ------------------------------------------------------------------
# DECISION: Drop identifier, constant, location, and leakage-prone columns.
# ------------------------------------------------------------------
# Decision:
#   Remove CustomerID, Count, Country, State, City, Zip Code, Lat Long,
#   Latitude, Longitude, Churn Label, Churn Score, and Churn Reason.
#   Keep only the modelling features + Churn Value (target).
#
# Alternatives considered:
#   1. Keep CustomerID as an index / join key.
#      Why not picked: CustomerID is a surrogate key with no predictive
#      signal and leaks row identity; automated retraining should be
#      reproducible from features alone, not from an arbitrary ID.
#   2. Keep Count (all values == 1).
#      Why not picked: constant column carries zero information and would
#      be dropped by any variance-based feature selector anyway.
#   3. Keep Country (single value "United States").
#      Why not picked: constant column, zero information.
#   4. Keep State / City / Zip Code / Latitude / Longitude as features.
#      Why not picked: high-cardinality location fields are not actionable
#      retention levers (you cannot easily move a customer), they inflate
#      one-hot dimensionality, and they risk overfitting to geography
#      rather than behaviour. They are useful for EDA/ops dashboards but
#      not for a retraining feature matrix.
#   5. Keep Churn Label (string "Yes"/"No").
#      Why not picked: redundant duplicate of the target Churn Value;
#      keeping both creates an exact-leakage feature.
#   6. Keep Churn Score (5-100 proprietary score).
#      Why not picked: likely a model output / target-derived score;
#      using it as a feature risks target leakage and makes retraining
#      depend on an opaque external score that may not be regenerated
#      consistently each retrain cycle.
#   7. Keep Churn Reason.
#      Why not picked: only populated for customers who already churned
#      (5,174 of 7,043 are null). It is a post-hoc explanation, not a
#      pre-churn predictor, and using it leaks the outcome.
#
# Columns retained for modelling:
#   Gender, Senior Citizen, Partner, Dependents, Tenure Months,
#   Phone Service, Multiple Lines, Internet Service, Online Security,
#   Online Backup, Device Protection, Tech Support, Streaming TV,
#   Streaming Movies, Contract, Paperless Billing, Payment Method,
#   Monthly Charges, Total Charges, Churn Value (target).
#
columns_to_drop = [
    "CustomerID", "Count", "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude", "Churn Label", "Churn Score",
    "Churn Reason",
]
df_model = df.drop(columns=columns_to_drop).copy()
print("Dropped columns:", columns_to_drop)
print("Remaining shape:", df_model.shape)
print("Remaining columns:", list(df_model.columns))


In [ ]:
# ------------------------------------------------------------------
# DECISION: Clean Total Charges (object -> numeric; whitespace -> NaN -> 0).
# ------------------------------------------------------------------
# Decision:
#   Convert Total Charges to numeric with errors="coerce" so the 11
#   whitespace cells become NaN, then impute those NaN with 0.
#
# Alternatives considered:
#   1. Leave Total Charges as object.
#      Why not picked: scikit-learn and most estimators require numeric
#      inputs; an object column breaks the retraining pipeline.
#   2. Drop the 11 rows with whitespace Total Charges.
#      Why not picked: all 11 have Tenure Months == 0 and Churn Value == 0
#      (brand-new customers not yet billed). Dropping them removes valid
#      non-churned examples and shrinks the majority class unnecessarily;
#      the information (tenure 0 => $0 total) is recoverable.
#   3. Impute Total Charges NaN with the median (~4527).
#      Why not picked: median imputation would fabricate a large historical
#      spend for customers with zero tenure, contradicting Tenure Months == 0
#      and biasing charge-based features upward.
#   4. Impute with Monthly Charges * Tenure Months.
#      Why not picked: for tenure == 0 this equals 0, identical to the
#      chosen option, but adds arithmetic complexity for no gain. Since
#      every NaN row has tenure 0, imputing 0 is exactly equivalent and
#      simpler to reproduce in retraining.
#
# Implementation note:
#   Whitespace-only strings become NaN under pd.to_numeric(..., errors="coerce").
#   Every NaN row has Tenure Months == 0, so imputing 0 is semantically
#   correct (no bills yet => $0 total charged).
#
total_charges_numeric = pd.to_numeric(df_model["Total Charges"], errors="coerce")
nan_mask = total_charges_numeric.isna()
print("Total Charges NaN after coercion:", int(nan_mask.sum()))
print("NaN rows all have Tenure Months == 0:", bool(
    df_model.loc[nan_mask, "Tenure Months"].eq(0).all()
) if nan_mask.any() else True)

df_model["Total Charges"] = total_charges_numeric.fillna(0)
print("Total Charges dtype after cleaning:", df_model["Total Charges"].dtype)
print("Remaining NaN in Total Charges:", int(df_model["Total Charges"].isna().sum()))


In [ ]:
# ------------------------------------------------------------------
# DECISION: Collapse 'No phone service' / 'No internet service' -> 'No'.
# ------------------------------------------------------------------
# Decision:
#   Map 'No phone service' -> 'No' in Multiple Lines, and
#   'No internet service' -> 'No' in the six online-service columns
#   (Online Security, Online Backup, Device Protection, Tech Support,
#   Streaming TV, Streaming Movies).
#
# Alternatives considered:
#   1. Keep 'No phone service' / 'No internet service' as distinct levels.
#      Why not picked: semantically they both mean "the customer does not
#      have this service". Keeping them separate inflates cardinality,
#      creates sparse one-hot columns, and splits a single behavioural
#      signal (absence of service) across multiple dummy variables,
#      weakening model learning.
#   2. Impute/'fill' with the mode of the column.
#      Why not picked: these are not missing values — they are legitimate
#      categorical states meaning "no service"; mode-imputation would
#      mislabel customers as having a service they do not.
#   3. Create a separate binary "has_internet_service" flag and keep
#      the original levels.
#      Why not picked: redundant — Internet Service column already encodes
#      this, and collapsing is simpler and more reproducible each retrain.
#
# After this mapping every affected column holds only "Yes"/"No",
# which allows binary 0/1 encoding in the next step.
#
service_cols = [
    "Multiple Lines",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
]
replacements = {
    "No phone service": "No",
    "No internet service": "No",
}
df_model[service_cols] = df_model[service_cols].replace(replacements)
print("Service columns after mapping:")
print(df_model[service_cols].apply(pd.Series.value_counts))


In [ ]:
# ------------------------------------------------------------------
# DECISION: Encode binary Yes/No columns to 1/0.
# ------------------------------------------------------------------
# Decision:
#   Map {"Yes": 1, "No": 0} on all columns that now hold only
#   "Yes"/"No": Senior Citizen, Partner, Dependents, Phone Service,
#   Paperless Billing, Multiple Lines, and the six online-service
#   columns (Online Security, Online Backup, Device Protection,
#   Tech Support, Streaming TV, Streaming Movies). Gender is mapped
#   {"Male": 1, "Female": 0}.
#
# Alternatives considered:
#   1. One-hot encode all binary columns.
#      Why not picked: for a strict binary feature one-hot creates a
#      redundant dummy pair (perfect collinearity with the intercept).
#      Label-encoding to 1/0 is more compact and tree/linear friendly.
#   2. Keep Senior Citizen as str "Yes"/"No".
#      Why not picked: stored as strings in the source; estimators need
#      numeric input, and 0/1 matches the semantic "is senior".
#   3. Keep the six online-service columns as multi-class one-hot.
#      Why not picked: after the service mapping they contain only
#      "Yes"/"No" (the "No internet service" level was collapsed),
#      so one-hot would add a redundant dummy column; binary 0/1 is
#      sufficient and more compact.
#   4. Use -1/+1 or keep as bool.
#      Why not picked: 0/1 is the conventional, library-agnostic choice
#      and integrates cleanly with one-hot of multi-class columns later.
#
binary_map = {"Yes": 1, "No": 0}
gender_map = {"Male": 1, "Female": 0}

binary_cols = [
    "Senior Citizen", "Partner", "Dependents", "Phone Service",
    "Paperless Billing", "Multiple Lines",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
]
for col in binary_cols:
    df_model[col] = df_model[col].map(binary_map)

df_model["Gender"] = df_model["Gender"].map(gender_map)
print("Binary-encoded columns:", binary_cols + ["Gender"])
print(df_model[binary_cols + ["Gender"]].head())


In [ ]:
# ------------------------------------------------------------------
# DECISION: One-hot encode the remaining genuinely multi-class columns.
# ------------------------------------------------------------------
# Decision:
#   One-hot encode (drop_first=False) the remaining categorical features
#   that still have more than two levels: Internet Service,
#   Contract, Payment Method.
#   All other columns are now numeric; the resulting dataframe is fully
#   numeric and ready for scikit-learn.
#
# Alternatives considered:
#   1. Ordinal / label encoding for nominal columns.
#      Why not picked: Internet Service, Contract, and Payment Method
#      have no natural ordering; ordinal encoding would impose a false
#      magnitude relationship and mislead linear/distance models.
#   2. drop_first=True (k-1 dummies).
#      Why not picked: drop_first avoids the dummy-trap only when a model
#      includes an intercept and the encoding must sum to a constant.
#      For automated retraining where algorithms vary (tree-based models
#      do not suffer from collinearity; some pipelines standardize),
#      keeping all levels is safer, more explicit, and makes every
#      category's contribution independently inspectable.
#   3. Target encoding / frequency encoding.
#      Why not picked: target encoding leaks target information into
#      features and must be re-fit carefully inside each retraining fold
#      to avoid leakage; frequency encoding adds instability for rare
#      levels. One-hot is leakage-free and deterministic.
#
nominal_cols = ["Internet Service", "Contract", "Payment Method"]
df_model = pd.get_dummies(df_model, columns=nominal_cols, prefix=nominal_cols, dtype=int)
print("One-hot encoded columns:", nominal_cols)
print("Shape after one-hot:", df_model.shape)
print("New columns:", [c for c in df_model.columns if any(n in c for n in nominal_cols)])


In [ ]:
# ------------------------------------------------------------------
# DECISION: Split features (X) and target (y); validate; save artefacts.
# ------------------------------------------------------------------
# Decision:
#   Target = Churn Value (0 = retained, 1 = churned). Features = all
#   remaining columns. Save cleaned dataframe, X, and y to CSV for the
#   automated retraining pipeline to consume.
#
# Alternatives considered:
#   1. Use Churn Label (string) as target instead of Churn Value.
#      Why not picked: Churn Value is already the numeric 0/1 encoding of
#      the same label; using Churn Label would require re-encoding inside
#      every retrain and adds a failure point.
#   2. Save only X and y, not the cleaned full dataframe.
#      Why not picked: the cleaned full dataframe is useful for auditing,
#      drift monitoring, and re-deriving splits; storing it costs little
#      and improves pipeline observability.
#   3. Train/test split here.
#      Why not picked: splitting belongs in the retraining/training step,
#      not the cleaning step, so that the retraining pipeline controls
#      stratification, random state, and leakage-safe partitioning.
#
assert "Churn Value" in df_model.columns, "Target column missing"
y = df_model["Churn Value"].astype(int)
X = df_model.drop(columns=["Churn Value"])

print("Target value counts:")
print(y.value_counts())
print("Feature shape X:", X.shape)
print("Target shape y:", y.shape)
print("Any NaN remaining in X:", bool(X.isna().any().any()))
print("Any NaN remaining in y:", bool(y.isna().any()))
print("X dtypes all numeric:", bool(((X.dtypes == "int64") | (X.dtypes == "float64")).all()))

# Save artefacts
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_model.to_csv(CLEAN_PATH, index=False)
X.to_csv(FEATURES_PATH, index=False)
y.to_csv(TARGET_PATH, index=False, header=["Churn Value"])
print()
print("Saved:")
print("  ", CLEAN_PATH)
print("  ", FEATURES_PATH)
print("  ", TARGET_PATH)


In [ ]:
# Final cleaning summary for the automated retraining pipeline
summary = {
    "source_rows": 7043,
    "source_columns": 33,
    "clean_rows": int(df_model.shape[0]),
    "clean_columns_incl_target": int(df_model.shape[1]),
    "feature_columns": int(X.shape[1]),
    "target_column": "Churn Value",
    "target_balance": y.value_counts().to_dict(),
    "dropped_columns": columns_to_drop,
    "total_charges_nan_imputed": int(nan_mask.sum()),
    "nan_remaining_X": int(X.isna().sum().sum()),
}
import json
print(json.dumps(summary, indent=2))
